In [1]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from datasets import load_dataset

# Load dataset
train_dataset = load_dataset("trl-lib/Capybara", split="train")

Generating test split: 100%|██████████| 200/200 [00:00<00:00, 15617.75 examples/s]


In [4]:
# View dataset
train_dataset

Dataset({
    features: ['source', 'messages', 'num_turns'],
    num_rows: 15806
})

In [5]:
# view sample dataset

train_dataset[0]

{'source': 'GPT4LLM',
 'messages': [{'content': 'Recommend a movie to watch.\n', 'role': 'user'},
  {'content': 'I would recommend the movie, "The Shawshank Redemption" which is a classic drama film starring Tim Robbins and Morgan Freeman. This film tells a powerful story about hope and resilience, as it follows the story of a young man who is wrongfully convicted of murder and sent to prison. Amidst the harsh realities of prison life, the protagonist forms a bond with a fellow inmate, and together they navigate the challenges of incarceration, while holding on to the hope of eventual freedom. This timeless movie is a must-watch for its moving performances, uplifting message, and unforgettable storytelling.',
   'role': 'assistant'},
  {'content': 'Describe the character development of Tim Robbins\' character in "The Shawshank Redemption".',
   'role': 'user'},
  {'content': 'In "The Shawshank Redemption", Tim Robbins plays the character of Andy Dufresne, a banker who is wrongfully con

In [6]:
train_dataset[2]

{'source': 'EverythingLM',
 'messages': [{'content': 'Considering that a film commences at 6:30 PM with a duration of 90 minutes, taking into account any potential intervals or breaks, at what precise moment will the cinematic experience reach its conclusion?',
   'role': 'user'},
  {'content': 'Step 1: Convert the duration of the film from minutes to hours and minutes.\n90 minutes = 1 hour and 30 minutes\n\nStep 2: Add the duration of the film to the starting time.\nStarting time: 6:30 PM\nDuration: 1 hour and 30 minutes\n\n6:30 PM + 1 hour = 7:30 PM\n7:30 PM + 30 minutes = 8:00 PM\n\nStep 3: Since there is no mention of intervals or breaks, we can assume that there are none. Therefore, the film will end at the calculated time.\n\nFinal Answer: The cinematic experience will reach its conclusion at 8:00 PM.',
   'role': 'assistant'},
  {'content': 'Considering the end time of the film and assuming the cinema is 45 minutes away from your home, what time should you leave your house to en

In [7]:
# load model
model_name = "Qwen/Qwen3-0.6B"

from transformers import AutoTokenizer, AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
model

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1476.35it/s]


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [15]:
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

train_args = SFTConfig(
    num_train_epochs=1,
    per_device_train_batch_size=32,
    # loss_type = "dft" # nll
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=train_args,
)
trainer.train()

Step,Training Loss
10,1.293135
20,1.281550
30,1.205888
40,1.313764
50,1.350390
60,1.341122


KeyboardInterrupt: 

### Use model after training

In [18]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

# pipe = pipeline("text-generation", model="Qwen3-0.6B-Instruct/checkpoint-5000")
pipe = pipeline("text-generation", model=model, tokenizer=AutoTokenizer.from_pretrained(model_name, trust_remote_code=True), device_map="auto", trust_remote_code=True)
prompt = "<|im_start|>user\nWhat is the capital of France? Answer in one word.<|im_end|>\n<|im_start|>assistant\n"
response = pipe(prompt)
response[0]["generated_text"]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 2055.46it/s]
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'<|im_start|>user\nWhat is the capital of France? Answer in one word.<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, the user is asking for the capital of France, and they want the answer in one word. Let me think. France\'s capital is Paris. But wait, the user specified "one word." So I need to make sure I\'m correct. I remember that Paris is indeed the capital. Maybe they just want a simple answer without any extra details. Let me double-check. Yes, Paris is the capital. No other city is considered the capital here. So the answer should be Paris.\n</think>\n\nParis'

Alternatively, use structured conversation format

In [ ]:
prompt = [{"role": "user", "content": "What is the capital of France? Answer in one word."}]
response = pipe(prompt)
response[0]["generated_text"]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': 'What is the capital of France? Answer in one word.'},
 {'role': 'assistant',
  'content': '<think>\nOkay, the user is asking for the capital of France and wants the answer in one word. Let me recall... France\'s capital is Paris. I need to make sure I\'m correct. Wait, is it Paris? Yeah, that\'s right. The capital city is called Paris. So the answer should be Paris. No other cities or words are needed here. Just "Paris" in one word.\n</think>\n\nParis'}]

: 

### Instruction Tuning example

In [ ]:
# Instruction tuning
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

trainer = SFTTrainer(
    model="Qwen/Qwen3-0.6B-Base",
    args=SFTConfig(
        output_dir="Qwen3-0.6B-Instruct",
        chat_template_path="HuggingFaceTB/SmolLM3-3B",
    ),
    train_dataset=load_dataset("trl-lib/Capybara", split="train"),
)
trainer.train()